# Free-features provider ensembles: top-6 tuned ML models

Локальный ноутбук для ансамблирования free-features экспериментов.

Для каждого провайдера отдельно:
- читается `llm_cluster_compare_results.csv`;
- для каждой ML-модели выбирается лучшая tuned-строка по `cv_MAE_internal`, затем берутся 6 лучших разных ML-моделей;
- для этих 6 моделей заново строятся prediction-базы на внутреннем blend validation и holdout;
- перебираются ансамбли: single, mean, median, trimmed mean, inverse-MAE weights, NNLS weights, constrained MAE weights, pair grid, greedy forward, stacking;
- лучший ансамбль выбирается по `selection_MAE`;
- результаты сохраняются в `artifacts_free_features_provider_ensembles_top6`.

In [ ]:
from pathlib import Path
import ast
import json
import math
import time
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd

from IPython.display import display

from scipy.optimize import minimize, nnls
from sklearn.base import clone
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import BayesianRidge, ElasticNet, HuberRegressor, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, pairwise_distances
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception as exc:
    XGB_AVAILABLE = False
    print('[WARN] xgboost is unavailable:', exc)

SEED = 2
TARGET_COL = 'duration_hours'
CAP_HOURS = 960
TOP_N_TUNED = 6
BLEND_TRAIN_FRAC = 0.85
BLEND_FIT_FRAC = 0.50
PAIR_WEIGHT_GRID = np.linspace(0.0, 1.0, 101)

ROOT = Path.cwd()
RUN_NAME = datetime.now().strftime('run_%Y%m%d_%H%M%S')
OUT_DIR = ROOT / 'artifacts_free_features_provider_ensembles_top6'
RUN_DIR = OUT_DIR / RUN_NAME
PRED_CACHE_DIR = RUN_DIR / 'pred_cache'
for sub in ['blend_val', 'fullfit_test_typical', 'fullfit_test_full']:
    (PRED_CACHE_DIR / sub).mkdir(parents=True, exist_ok=True)

PROVIDER_SPECS = {
    'gpt': {
        'artifact_dir': ROOT / 'llm_local_runs' / 'gpt_api_free_features' / 'artifacts_gpt_api_free_features',
        'label': 'GPT free features',
    },
    'claude': {
        'artifact_dir': ROOT / 'llm_local_runs' / 'claude_api_free_features' / 'artifacts_claude_api_free_features',
        'label': 'Claude free features',
    },
    'qwen': {
        'artifact_dir': ROOT / 'qwen_local_free_features' / 'artifacts_qwen_local_free_features',
        'label': 'Qwen free features',
    },
    'ollama': {
        'artifact_dir': ROOT / 'ollama_local_free_features' / 'artifacts_ollama_local_free_features',
        'label': 'Ollama free features',
    },
}
for spec in PROVIDER_SPECS.values():
    spec['summary_path'] = spec['artifact_dir'] / 'llm_cluster_compare_results.csv'

DATA_CANDIDATES = [
    ROOT / 'data_finall_without_TTM.csv',
    ROOT / 'llm_local_runs' / 'gpt_api_free_features' / 'inputs' / 'data_finall_without_TTM.csv',
    ROOT / 'llm_local_runs' / 'claude_api_free_features' / 'inputs' / 'data_finall_without_TTM.csv',
    ROOT / 'qwen_local' / 'inputs' / 'data_finall_without_TTM.csv',
    ROOT / 'ollama_local' / 'inputs' / 'data_finall_without_TTM.csv',
    ROOT / 'data_finall.csv',
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('data_finall_without_TTM.csv / data_finall.csv not found')

print('ROOT:', ROOT)
print('DATA_PATH:', DATA_PATH)
print('OUT_DIR:', OUT_DIR)
print('RUN_DIR:', RUN_DIR)
print('Providers:')
for provider, spec in PROVIDER_SPECS.items():
    print(f"  {provider:<7s} {spec['artifact_dir']}")

In [ ]:
def read_dataset(path: Path):
    df = pd.read_csv(path)
    if TARGET_COL not in df.columns:
        raise ValueError(f'{TARGET_COL} is absent in {path}')
    drop_cols = ['Ключ', 'Задача', 'Статус', 'Резолюция', 'Создано', 'Дата завершения', 'created_dt']
    return df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

def normalize_X(df_in):
    out = df_in.copy()
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0)
    for c in out.columns:
        if out[c].dtype == 'bool':
            out[c] = out[c].astype(int)
    return out

def metrics(y_true, pred):
    pred = np.clip(np.asarray(pred, dtype=float), 0, None)
    return {
        'MAE': float(mean_absolute_error(y_true, pred)),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, pred))),
        'MdAE': float(median_absolute_error(y_true, pred)),
    }

def parse_params(raw):
    if isinstance(raw, dict):
        params = raw
    elif raw is None:
        params = {}
    else:
        try:
            if pd.isna(raw):
                return {}
        except Exception:
            pass
        text = str(raw).strip()
        if not text or text.lower() in {'nan', 'none', 'null'}:
            return {}
        try:
            params = json.loads(text)
        except Exception:
            text2 = text.replace('NaN', 'None').replace('nan', 'None').replace('null', 'None')
            text2 = text2.replace('true', 'True').replace('false', 'False')
            params = ast.literal_eval(text2)
    out = {}
    for k, v in dict(params).items():
        if hasattr(v, 'item'):
            v = v.item()
        out[k] = v
    return out

def is_nan_like(v):
    try:
        return bool(pd.isna(v))
    except Exception:
        return False

def apply_params_to_pipeline(pipe, params):
    pipe = clone(pipe)
    names = pipe.get_params(deep=True)
    settable = {}
    for k, v in params.items():
        # Skip null/NaN defaults, especially XGBoost optional params.
        if v is None or is_nan_like(v):
            continue
        if f'm__{k}' in names:
            settable[f'm__{k}'] = v
        elif k in names:
            settable[k] = v
    if settable:
        pipe.set_params(**settable)
    return pipe

def model_template(model_name):
    if model_name == 'Ridge':
        return Pipeline([('sc', StandardScaler()), ('m', Ridge(random_state=SEED))])
    if model_name == 'Lasso':
        return Pipeline([('sc', StandardScaler()), ('m', Lasso(random_state=SEED, max_iter=100000))])
    if model_name == 'ElasticNet':
        return Pipeline([('sc', StandardScaler()), ('m', ElasticNet(random_state=SEED, max_iter=100000))])
    if model_name == 'HuberReg':
        return Pipeline([('sc', StandardScaler()), ('m', HuberRegressor(max_iter=10000))])
    if model_name == 'BayRidge':
        return Pipeline([('sc', StandardScaler()), ('m', BayesianRidge())])
    if model_name == 'RF':
        return Pipeline([('m', RandomForestRegressor(random_state=SEED, n_jobs=-1))])
    if model_name == 'ExtraTrees':
        return Pipeline([('m', ExtraTreesRegressor(random_state=SEED, n_jobs=-1))])
    if model_name == 'GBoost':
        return Pipeline([('m', GradientBoostingRegressor(random_state=SEED))])
    if model_name == 'XGB':
        if not XGB_AVAILABLE:
            raise ImportError('xgboost is required for XGB rows')
        return Pipeline([('m', XGBRegressor(objective='reg:absoluteerror', eval_metric='mae', random_state=SEED, tree_method='hist', n_jobs=-1, verbosity=0))])
    if model_name == 'KNN':
        return Pipeline([('sc', StandardScaler()), ('m', KNeighborsRegressor())])
    raise ValueError(f'Unknown model: {model_name}')

df = read_dataset(DATA_PATH)
split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy()
test_typical = test_full[test_full[TARGET_COL] < CAP_HOURS].copy()
train_filt = train_full[train_full[TARGET_COL] < CAP_HOURS].copy()

meta_Xtr0 = train_filt.drop(columns=[TARGET_COL], errors='ignore').reset_index(drop=True)
meta_ytr0 = train_filt[TARGET_COL].reset_index(drop=True)
meta_Xte0 = test_full.drop(columns=[TARGET_COL], errors='ignore').reset_index(drop=True)
meta_yte0 = test_full[TARGET_COL].reset_index(drop=True)
meta_Xte_typ0 = test_typical.drop(columns=[TARGET_COL], errors='ignore').reset_index(drop=True)
meta_yte_typ0 = test_typical[TARGET_COL].reset_index(drop=True)

test_typical_mask = (test_full[TARGET_COL].reset_index(drop=True) < CAP_HOURS)

print('train_full  ', train_full.shape)
print('train_filt  ', train_filt.shape)
print('test_full   ', test_full.shape)
print('test_typical', test_typical.shape)

In [ ]:
def make_clusterer(name, params):
    if name == 'MiniBatchKMeans':
        return MiniBatchKMeans(n_clusters=int(params['n_clusters']), random_state=SEED, n_init=20, batch_size=1024)
    if name == 'GaussianMixture':
        return GaussianMixture(
            n_components=int(params['n_components']),
            covariance_type=params.get('covariance_type', 'diag'),
            random_state=SEED,
        )
    raise ValueError(name)

def probs_from_dist(all_d):
    inv = 1.0 / (all_d + 1e-6)
    return inv / inv.sum(axis=1, keepdims=True)

def fit_clusterer_and_assign(name, params, Xtr, Xte):
    est = make_clusterer(name, params)
    if name == 'MiniBatchKMeans':
        est.fit(Xtr)
        tr_labels = est.predict(Xtr).astype(int)
        te_labels = est.predict(Xte).astype(int)
        tr_d = pairwise_distances(Xtr, est.cluster_centers_)
        te_d = pairwise_distances(Xte, est.cluster_centers_)
        return tr_labels, te_labels, probs_from_dist(tr_d), probs_from_dist(te_d)
    if name == 'GaussianMixture':
        est.fit(Xtr)
        tr_proba = est.predict_proba(Xtr)
        te_proba = est.predict_proba(Xte)
        return np.argmax(tr_proba, axis=1).astype(int), np.argmax(te_proba, axis=1).astype(int), tr_proba, te_proba
    raise ValueError(name)

def build_cluster_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    tr_feat = pd.DataFrame({'cluster_id': tr_labels})
    te_feat = pd.DataFrame({'cluster_id': te_labels})
    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat['cluster_size_train'] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat['cluster_size_train'] = pd.Series(te_labels).map(tr_sizes).fillna(0)
    tr_feat['is_noise'] = (tr_feat['cluster_id'] == -1).astype(int)
    te_feat['is_noise'] = (te_feat['cluster_id'] == -1).astype(int)

    tr_ohe = pd.get_dummies(tr_feat['cluster_id'], prefix='cluster')
    te_ohe = pd.get_dummies(te_feat['cluster_id'], prefix='cluster').reindex(columns=tr_ohe.columns, fill_value=0)
    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)

    if tr_proba is not None and te_proba is not None:
        for k in range(tr_proba.shape[1]):
            tr_feat[f'cluster_proba_{k}'] = tr_proba[:, k]
            te_feat[f'cluster_proba_{k}'] = te_proba[:, k]
    return tr_feat.reset_index(drop=True), te_feat.reset_index(drop=True)

FIXED_CLUSTER_CONFIGS = {
    'gmm_diag_5': {'clusterer': 'GaussianMixture', 'params': {'n_components': 5, 'covariance_type': 'diag'}},
    'mbkm_2': {'clusterer': 'MiniBatchKMeans', 'params': {'n_clusters': 2}},
}

raw_scaler = StandardScaler()
raw_Xtr_scaled = raw_scaler.fit_transform(meta_Xtr0)
raw_Xte_scaled = raw_scaler.transform(meta_Xte0)
raw_pca = PCA(n_components=min(30, meta_Xtr0.shape[1]), random_state=SEED)
raw_Xtr_embed = raw_pca.fit_transform(raw_Xtr_scaled)
raw_Xte_embed = raw_pca.transform(raw_Xte_scaled)

raw_cluster_features = {}
for tag, cfg in FIXED_CLUSTER_CONFIGS.items():
    tr_labels, te_labels, tr_proba, te_proba = fit_clusterer_and_assign(cfg['clusterer'], cfg['params'], raw_Xtr_embed, raw_Xte_embed)
    raw_cluster_features[tag] = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)
print('cluster tags:', list(raw_cluster_features))

def find_feature_file(artifact_dir: Path, experiment: str, split: str) -> Path:
    patterns = [
        f'llm_features_{experiment}_{split}.csv',
        f'llm_features_*_{experiment}_{split}.csv',
    ]
    matches = []
    for pattern in patterns:
        matches.extend(sorted(artifact_dir.glob(pattern)))
    matches = sorted(set(matches), key=lambda x: (len(x.name), x.name))
    if not matches:
        raise FileNotFoundError(f'No feature file for {experiment}/{split} in {artifact_dir}; tried {patterns}')
    return matches[-1]

def load_llm_features(artifact_dir: Path, experiment: str):
    train_path = find_feature_file(artifact_dir, experiment, 'train')
    test_path = find_feature_file(artifact_dir, experiment, 'test')
    tr = pd.read_csv(train_path).drop(columns=['row_id'], errors='ignore').reset_index(drop=True)
    te = pd.read_csv(test_path).drop(columns=['row_id'], errors='ignore').reset_index(drop=True)
    if len(tr) != len(meta_Xtr0):
        raise ValueError(f'{train_path} rows={len(tr)} expected={len(meta_Xtr0)}')
    if len(te) != len(meta_Xte0):
        raise ValueError(f'{test_path} rows={len(te)} expected={len(meta_Xte0)}')
    return tr, te, train_path, test_path

_FEATURE_CACHE = {}

def build_feature_space(provider: str, experiment: str):
    key = (provider, experiment)
    if key in _FEATURE_CACHE:
        return _FEATURE_CACHE[key]

    artifact_dir = PROVIDER_SPECS[provider]['artifact_dir']
    if experiment == 'llm_only':
        llm_tr, llm_te, tr_path, te_path = load_llm_features(artifact_dir, 'llm_only')
        Xtr = pd.concat([meta_Xtr0.reset_index(drop=True), llm_tr], axis=1)
        Xte_full = pd.concat([meta_Xte0.reset_index(drop=True), llm_te], axis=1)
    elif experiment.startswith('cluster_before_llm__'):
        tag = experiment.replace('cluster_before_llm__', '')
        llm_tr, llm_te, tr_path, te_path = load_llm_features(artifact_dir, experiment)
        cl_tr, cl_te = raw_cluster_features[tag]
        Xtr = pd.concat([meta_Xtr0.reset_index(drop=True), cl_tr, llm_tr], axis=1)
        Xte_full = pd.concat([meta_Xte0.reset_index(drop=True), cl_te, llm_te], axis=1)
    elif experiment.startswith('llm_then_cluster__'):
        tag = experiment.replace('llm_then_cluster__', '')
        llm_tr, llm_te, tr_path, te_path = load_llm_features(artifact_dir, 'llm_only')
        Xtr_llm = pd.concat([meta_Xtr0.reset_index(drop=True), llm_tr], axis=1)
        Xte_llm = pd.concat([meta_Xte0.reset_index(drop=True), llm_te], axis=1)
        cfg = FIXED_CLUSTER_CONFIGS[tag]
        scaler = StandardScaler()
        Xtr_scaled = scaler.fit_transform(normalize_X(Xtr_llm))
        Xte_scaled = scaler.transform(normalize_X(Xte_llm))
        pca = PCA(n_components=min(30, Xtr_llm.shape[1]), random_state=SEED)
        Xtr_embed = pca.fit_transform(Xtr_scaled)
        Xte_embed = pca.transform(Xte_scaled)
        tr_labels, te_labels, tr_proba, te_proba = fit_clusterer_and_assign(cfg['clusterer'], cfg['params'], Xtr_embed, Xte_embed)
        cl_tr, cl_te = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)
        Xtr = pd.concat([meta_Xtr0.reset_index(drop=True), llm_tr, cl_tr], axis=1)
        Xte_full = pd.concat([meta_Xte0.reset_index(drop=True), llm_te, cl_te], axis=1)
    else:
        raise ValueError(experiment)

    Xtr = normalize_X(Xtr)
    Xte_full = normalize_X(Xte_full)
    Xte_typ = Xte_full.loc[test_typical_mask.values].reset_index(drop=True)
    payload = (Xtr, Xte_typ, Xte_full, str(tr_path), str(te_path))
    _FEATURE_CACHE[key] = payload
    return payload

In [ ]:
def select_top6_tuned_rows(provider: str):
    path = PROVIDER_SPECS[provider]['summary_path']
    dfp = pd.read_csv(path)
    required = {'model', 'experiment', 'cv_MAE_internal', 'best_params'}
    missing = required - set(dfp.columns)
    if missing:
        raise ValueError(f'{path} missing columns: {missing}')
    dfp = dfp.copy()
    dfp['provider'] = provider

    # Important: top-6 means six different ML model families.
    # First take the best tuned scheme for each ML model, then keep the six best models.
    best_per_model = (
        dfp.sort_values(['model', 'cv_MAE_internal', 'holdout_typical_MAE', 'holdout_full_MAE'])
           .groupby('model', as_index=False)
           .head(1)
           .reset_index(drop=True)
    )
    top = (
        best_per_model.sort_values(['cv_MAE_internal', 'holdout_typical_MAE', 'holdout_full_MAE'])
                      .head(TOP_N_TUNED)
                      .reset_index(drop=True)
    )
    top['rank_in_provider'] = np.arange(1, len(top) + 1)
    return top

provider_top6_rows = pd.concat([select_top6_tuned_rows(p) for p in PROVIDER_SPECS], ignore_index=True)
provider_top6_rows.to_csv(RUN_DIR / 'provider_top6_tuned_rows.csv', index=False)
provider_top6_rows.to_csv(OUT_DIR / 'provider_top6_tuned_rows_latest.csv', index=False)

display_cols = ['provider', 'rank_in_provider', 'model', 'experiment', 'cv_MAE_internal', 'holdout_typical_MAE', 'holdout_full_MAE']
display(provider_top6_rows[display_cols])

def fit_one_row(row):
    provider = row['provider']
    model_name = row['model']
    experiment = row['experiment']
    params = parse_params(row['best_params'])
    Xtr, Xte_typ, Xte_full, train_feature_path, test_feature_path = build_feature_space(provider, experiment)

    n = len(Xtr)
    blend_start = int(n * BLEND_TRAIN_FRAC)
    X_blend_train = Xtr.iloc[:blend_start].reset_index(drop=True)
    y_blend_train = meta_ytr0.iloc[:blend_start].reset_index(drop=True)
    X_blend_val = Xtr.iloc[blend_start:].reset_index(drop=True)
    y_blend_val = meta_ytr0.iloc[blend_start:].reset_index(drop=True)

    pipe = apply_params_to_pipeline(model_template(model_name), params)
    pipe.fit(X_blend_train, y_blend_train)
    pred_blend = np.clip(pipe.predict(X_blend_val), 0, None)

    full_pipe = apply_params_to_pipeline(model_template(model_name), params)
    full_pipe.fit(Xtr, meta_ytr0)
    pred_typ = np.clip(full_pipe.predict(Xte_typ), 0, None)
    pred_full = np.clip(full_pipe.predict(Xte_full), 0, None)

    base_id = f'{provider}::rank{int(row["rank_in_provider"]):02d}::{experiment}::{model_name}'
    np.save(PRED_CACHE_DIR / 'blend_val' / f'{base_id.replace("::", "__")}.npy', pred_blend)
    np.save(PRED_CACHE_DIR / 'fullfit_test_typical' / f'{base_id.replace("::", "__")}.npy', pred_typ)
    np.save(PRED_CACHE_DIR / 'fullfit_test_full' / f'{base_id.replace("::", "__")}.npy', pred_full)

    return {
        'base_id': base_id,
        'provider': provider,
        'rank_in_provider': int(row['rank_in_provider']),
        'model': model_name,
        'experiment': experiment,
        'source_cv_MAE_internal': float(row['cv_MAE_internal']),
        'source_holdout_typical_MAE': float(row['holdout_typical_MAE']),
        'source_holdout_full_MAE': float(row['holdout_full_MAE']),
        'train_feature_path': train_feature_path,
        'test_feature_path': test_feature_path,
        'pred_blend_val': pred_blend,
        'pred_typical': pred_typ,
        'pred_full': pred_full,
        'single_blend_MAE': metrics(y_blend_val, pred_blend)['MAE'],
        'single_typical_MAE': metrics(meta_yte_typ0, pred_typ)['MAE'],
        'single_full_MAE': metrics(meta_yte0, pred_full)['MAE'],
    }

base_payloads = []
t0 = time.time()
for _, row in provider_top6_rows.iterrows():
    print(f"fit {row['provider']:<7s} rank={int(row['rank_in_provider'])} {row['model']:<10s} {row['experiment']}")
    base_payloads.append(fit_one_row(row))
    print(f"  single_blend_MAE={base_payloads[-1]['single_blend_MAE']:.4f} typical={base_payloads[-1]['single_typical_MAE']:.4f} full={base_payloads[-1]['single_full_MAE']:.4f}")
print('elapsed min:', round((time.time() - t0) / 60, 2))

base_meta = pd.DataFrame([{k: v for k, v in p.items() if not k.startswith('pred_')} for p in base_payloads])
base_meta.to_csv(RUN_DIR / 'refit_top6_base_metrics.csv', index=False)
base_meta.to_csv(OUT_DIR / 'refit_top6_base_metrics_latest.csv', index=False)
display(base_meta.sort_values(['provider', 'single_blend_MAE']))

In [ ]:
def weighted_average(preds, weights):
    mat = np.vstack([np.asarray(p, dtype=float) for p in preds])
    w = np.asarray(weights, dtype=float)
    if np.allclose(w.sum(), 0):
        w = np.ones_like(w)
    w = w / w.sum()
    return np.clip(np.average(mat, axis=0, weights=w), 0, None)

def trimmed_mean(preds, trim=0.10):
    mat = np.sort(np.vstack([np.asarray(p, dtype=float) for p in preds]), axis=0)
    n = mat.shape[0]
    k = int(np.floor(n * trim))
    if 2 * k >= n:
        return np.clip(np.mean(mat, axis=0), 0, None)
    return np.clip(np.mean(mat[k:n-k], axis=0), 0, None)

def split_blend_targets():
    yb = meta_ytr0.iloc[int(len(meta_ytr0) * BLEND_TRAIN_FRAC):].reset_index(drop=True)
    cut = int(len(yb) * BLEND_FIT_FRAC)
    return yb.iloc[:cut].reset_index(drop=True), yb.iloc[cut:].reset_index(drop=True), cut

def stacker_templates():
    stackers = {
        'stack_linear_positive': LinearRegression(positive=True),
        'stack_ridge_a1': Ridge(alpha=1.0, random_state=SEED),
        'stack_bayridge': BayesianRidge(),
        'stack_huber': HuberRegressor(max_iter=10000),
        'stack_gboost': GradientBoostingRegressor(random_state=SEED, loss='absolute_error', n_estimators=80, max_depth=2, learning_rate=0.05),
        'stack_extratrees': ExtraTreesRegressor(random_state=SEED, n_estimators=200, min_samples_leaf=8, n_jobs=-1),
    }
    return stackers

def add_row(rows, provider, family, name, combo, weights, pred_sel, pred_typ, pred_full, extra=None):
    ids = [p['base_id'] for p in combo]
    weights_list = [] if weights is None else [float(x) for x in weights]
    if weights is None or len(weights_list) == 0:
        effective_n_models = len(combo)
    else:
        effective_n_models = int(np.sum(np.abs(np.asarray(weights_list, dtype=float)) > 1e-6))
    row = {
        'provider': provider,
        'family': family,
        'name': name,
        'n_models': len(combo),
        'effective_n_models': effective_n_models,
        'models': json.dumps(ids, ensure_ascii=False),
        'weights': json.dumps(weights_list, ensure_ascii=False),
        'selection_MAE': metrics(y_sel, pred_sel)['MAE'],
        'selection_RMSE': metrics(y_sel, pred_sel)['RMSE'],
        'selection_MdAE': metrics(y_sel, pred_sel)['MdAE'],
        'MAE_typical': metrics(meta_yte_typ0, pred_typ)['MAE'],
        'RMSE_typical': metrics(meta_yte_typ0, pred_typ)['RMSE'],
        'MdAE_typical': metrics(meta_yte_typ0, pred_typ)['MdAE'],
        'MAE_full': metrics(meta_yte0, pred_full)['MAE'],
        'RMSE_full': metrics(meta_yte0, pred_full)['RMSE'],
        'MdAE_full': metrics(meta_yte0, pred_full)['MdAE'],
    }
    if extra:
        row.update(extra)
    rows.append(row)

def candidate_ensembles_for_provider(payloads):
    global y_fit, y_sel
    rows = []
    payloads = list(payloads)
    y_fit, y_sel, cut = split_blend_targets()

    def split_pred(arr):
        arr = np.asarray(arr, dtype=float)
        return arr[:cut], arr[cut:]

    # Single models.
    for pld in payloads:
        _, pred_sel = split_pred(pld['pred_blend_val'])
        add_row(rows, pld['provider'], 'single', 'single', [pld], [1.0], pred_sel, pld['pred_typical'], pld['pred_full'])

    # Full-pool simple aggregates.
    provider = payloads[0]['provider']
    full_combo = payloads
    full_sel_preds = [split_pred(p['pred_blend_val'])[1] for p in full_combo]
    add_row(rows, provider, 'aggregate', 'top6_mean', full_combo, [1 / len(full_combo)] * len(full_combo),
            np.mean(full_sel_preds, axis=0), np.mean([p['pred_typical'] for p in full_combo], axis=0), np.mean([p['pred_full'] for p in full_combo], axis=0))
    add_row(rows, provider, 'aggregate', 'top6_median', full_combo, None,
            np.median(full_sel_preds, axis=0), np.median([p['pred_typical'] for p in full_combo], axis=0), np.median([p['pred_full'] for p in full_combo], axis=0))
    add_row(rows, provider, 'aggregate', 'top6_trimmed_mean_10', full_combo, None,
            trimmed_mean(full_sel_preds, 0.10), trimmed_mean([p['pred_typical'] for p in full_combo], 0.10), trimmed_mean([p['pred_full'] for p in full_combo], 0.10))

    # Exhaustive subsets with mean, inverse-MAE, NNLS, constrained MAE weights.
    for r in range(2, len(payloads) + 1):
        for combo in combinations(payloads, r):
            combo = list(combo)
            ids = [p['base_id'] for p in combo]
            fit_preds = [split_pred(p['pred_blend_val'])[0] for p in combo]
            sel_preds = [split_pred(p['pred_blend_val'])[1] for p in combo]
            typ_preds = [p['pred_typical'] for p in combo]
            full_preds = [p['pred_full'] for p in combo]

            w_mean = [1 / r] * r
            add_row(rows, provider, 'subset_mean', f'subset{r}_mean', combo, w_mean,
                    weighted_average(sel_preds, w_mean), weighted_average(typ_preds, w_mean), weighted_average(full_preds, w_mean))

            single_err = np.asarray([metrics(y_fit, pred)['MAE'] for pred in fit_preds])
            inv_w = 1 / np.maximum(single_err, 1e-9)
            inv_w = inv_w / inv_w.sum()
            add_row(rows, provider, 'subset_inverse', f'subset{r}_inverse_fit_mae', combo, inv_w,
                    weighted_average(sel_preds, inv_w), weighted_average(typ_preds, inv_w), weighted_average(full_preds, inv_w))

            A_fit = np.vstack(fit_preds).T
            nnls_w, _ = nnls(A_fit, y_fit)
            if np.allclose(nnls_w.sum(), 0):
                nnls_w = np.ones(r) / r
            else:
                nnls_w = nnls_w / nnls_w.sum()
            add_row(rows, provider, 'subset_nnls', f'subset{r}_nnls', combo, nnls_w,
                    weighted_average(sel_preds, nnls_w), weighted_average(typ_preds, nnls_w), weighted_average(full_preds, nnls_w))

            def obj(w):
                return mean_absolute_error(y_fit, weighted_average(fit_preds, w))
            cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},)
            bounds = [(0.0, 1.0)] * r
            res = minimize(obj, np.ones(r) / r, method='SLSQP', bounds=bounds, constraints=cons, options={'maxiter': 300, 'ftol': 1e-8})
            opt_w = res.x if res.success else np.ones(r) / r
            opt_w = np.clip(opt_w, 0, None)
            opt_w = opt_w / opt_w.sum() if opt_w.sum() > 0 else np.ones(r) / r
            add_row(rows, provider, 'subset_constrained_mae', f'subset{r}_constrained_mae', combo, opt_w,
                    weighted_average(sel_preds, opt_w), weighted_average(typ_preds, opt_w), weighted_average(full_preds, opt_w),
                    extra={'optimizer_success': bool(res.success)})

            if r == 2:
                best = None
                for w0 in PAIR_WEIGHT_GRID:
                    ws = np.array([float(w0), float(1 - w0)])
                    fit_mae = metrics(y_fit, weighted_average(fit_preds, ws))['MAE']
                    if best is None or fit_mae < best[0]:
                        best = (fit_mae, ws)
                ws = best[1]
                add_row(rows, provider, 'pair_grid', 'pair_grid_fit_mae', combo, ws,
                        weighted_average(sel_preds, ws), weighted_average(typ_preds, ws), weighted_average(full_preds, ws),
                        extra={'pair_fit_MAE': float(best[0])})

    # Greedy forward selection using mean aggregation on fit split.
    remaining = list(payloads)
    current = []
    while remaining:
        best = None
        for cand in remaining:
            trial = current + [cand]
            fit_preds = [split_pred(p['pred_blend_val'])[0] for p in trial]
            score = metrics(y_fit, np.mean(fit_preds, axis=0))['MAE']
            if best is None or score < best[0]:
                best = (score, cand)
        current.append(best[1])
        remaining = [p for p in remaining if p is not best[1]]
        if len(current) >= 2:
            sel_preds = [split_pred(p['pred_blend_val'])[1] for p in current]
            add_row(rows, provider, 'greedy', f'greedy_mean_{len(current)}', current, [1 / len(current)] * len(current),
                    np.mean(sel_preds, axis=0), np.mean([p['pred_typical'] for p in current], axis=0), np.mean([p['pred_full'] for p in current], axis=0),
                    extra={'greedy_fit_MAE': float(best[0])})

    # Stackers on all top-6 predictions.
    X_fit_stack = np.vstack([split_pred(p['pred_blend_val'])[0] for p in payloads]).T
    X_sel_stack = np.vstack([split_pred(p['pred_blend_val'])[1] for p in payloads]).T
    X_typ_stack = np.vstack([p['pred_typical'] for p in payloads]).T
    X_full_stack = np.vstack([p['pred_full'] for p in payloads]).T
    for name, stacker in stacker_templates().items():
        try:
            st = clone(stacker)
            st.fit(X_fit_stack, y_fit)
            pred_sel = np.clip(st.predict(X_sel_stack), 0, None)
            pred_typ = np.clip(st.predict(X_typ_stack), 0, None)
            pred_full = np.clip(st.predict(X_full_stack), 0, None)
            weights = getattr(st, 'coef_', None)
            if weights is not None:
                weights = np.asarray(weights).ravel().tolist()
            add_row(rows, provider, 'stacking', name, payloads, weights, pred_sel, pred_typ, pred_full)
        except Exception as exc:
            rows.append({
                'provider': provider,
                'family': 'stacking_failed',
                'name': name,
                'n_models': len(payloads),
                'models': json.dumps([p['base_id'] for p in payloads], ensure_ascii=False),
                'weights': json.dumps([]),
                'selection_MAE': np.inf,
                'MAE_typical': np.inf,
                'MAE_full': np.inf,
                'error': str(exc),
            })

    return pd.DataFrame(rows)

leaderboards = []
for provider in PROVIDER_SPECS:
    payloads = [p for p in base_payloads if p['provider'] == provider]
    print(f'build ensembles for {provider}: {len(payloads)} base models')
    lb = candidate_ensembles_for_provider(payloads)
    lb = lb.sort_values(['selection_MAE', 'MAE_typical', 'MAE_full']).reset_index(drop=True)
    lb.to_csv(RUN_DIR / f'{provider}_ensemble_leaderboard.csv', index=False)
    lb.to_csv(OUT_DIR / f'{provider}_ensemble_leaderboard_latest.csv', index=False)
    leaderboards.append(lb)

all_leaderboard = pd.concat(leaderboards, ignore_index=True).sort_values(['provider', 'selection_MAE', 'MAE_typical', 'MAE_full']).reset_index(drop=True)
all_leaderboard.to_csv(RUN_DIR / 'all_provider_ensemble_leaderboard.csv', index=False)
all_leaderboard.to_csv(OUT_DIR / 'all_provider_ensemble_leaderboard_latest.csv', index=False)

best_provider_ensembles = (
    all_leaderboard[(all_leaderboard['n_models'] >= 2) & (all_leaderboard['effective_n_models'] >= 2)]
    .sort_values(['provider', 'selection_MAE', 'MAE_typical', 'MAE_full'])
    .groupby('provider', as_index=False)
    .head(1)
    .reset_index(drop=True)
)
best_provider_ensembles.to_csv(RUN_DIR / 'best_provider_ensembles.csv', index=False)
best_provider_ensembles.to_csv(OUT_DIR / 'best_provider_ensembles_latest.csv', index=False)

print('Best provider-only free-feature ensembles from top-6 tuned rows:')
display(best_provider_ensembles)
print('Saved to:', RUN_DIR)

In [ ]:
summary = {
    'run_name': RUN_NAME,
    'top_n_tuned': TOP_N_TUNED,
    'blend_train_frac': BLEND_TRAIN_FRAC,
    'blend_fit_frac': BLEND_FIT_FRAC,
    'data_path': str(DATA_PATH),
    'out_dir': str(OUT_DIR),
    'run_dir': str(RUN_DIR),
    'providers': {k: str(v['artifact_dir']) for k, v in PROVIDER_SPECS.items()},
    'best_provider_ensembles': best_provider_ensembles.to_dict(orient='records'),
}
with open(RUN_DIR / 'run_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
with open(OUT_DIR / 'run_manifest_latest.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

compact = best_provider_ensembles[[
    'provider', 'family', 'name', 'n_models', 'effective_n_models', 'selection_MAE', 'MAE_typical', 'MAE_full', 'models', 'weights'
]].copy()
compact.to_csv(OUT_DIR / 'compact_best_provider_ensembles_latest.csv', index=False)
display(compact)
print('Latest files:')
for p in sorted(OUT_DIR.glob('*_latest.*')):
    print(' ', p)